# Tutorial: single level adaptive sampling

First we import the objects from the `exauq` package.

In [1]:
from exauq.core.emulators import MogpEmulator  # A Gaussian process emulator backed by mogp-emulator
from exauq.core.modelling import (
    TrainingDatum,  # For working with emulator training data
    SimulatorDomain,  # For defining the input space of the emulator
)
from exauq.core.designers import compute_single_level_loo_samples  # Function for performing adaptive sampling

# Don't display warnings from mogp-emulator
import warnings
warnings.filterwarnings("ignore")

## Loading in training data

Read in some training data for the GP from a pre-prepared csv file:

In [2]:
training_data = TrainingDatum.read_from_csv("./data/gp_training_data.csv", header = True)
training_data

(TrainingDatum(input=Input(0.0608263616598308, 0.2937336532118139), output=-1.0305006038593505),
 TrainingDatum(input=Input(0.9234380600806524, 0.5983389693269525), output=0.3948727799746161),
 TrainingDatum(input=Input(0.19704926841400994, 0.44906762812658374), output=-0.7246557854327791),
 TrainingDatum(input=Input(0.0804580190777664, 0.5227155982309132), output=-0.6117928746743624),
 TrainingDatum(input=Input(0.9795433844227166, 0.5099688760976656), output=0.31532881027919646),
 TrainingDatum(input=Input(0.7108487037945828, 0.40836831865933987), output=-0.33377468034266067),
 TrainingDatum(input=Input(0.3825924172249514, 0.9784441682423706), output=0.667960553954811),
 TrainingDatum(input=Input(0.46051831260535014, 0.7971499197373327), output=0.2304614681463526),
 TrainingDatum(input=Input(0.8641727138094001, 0.7393597021445757), output=0.6185933882194996),
 TrainingDatum(input=Input(0.14372464346306613, 0.6069954270764075), output=-0.4181179136664317))

## Train a GP

Next we'll train a GP. We first create a new GP using the `MogpEmulator` class, specifying that it uses a Matern 5/2 kernel.

In [3]:
gp = MogpEmulator(kernel="Matern52")

Too few unique inputs; defaulting to flat priors
Too few unique inputs; defaulting to flat priors


Note that this GP hasn't been trained on any data yet; we can verify this be examining its `training_data` property:

In [4]:
gp.training_data

()

Now we'll train it on the data we read in earlier.

In [5]:
gp.fit(training_data)

## Find design point(s) using LOO adaptive sampling

Let's now find a new design point using the leave-one-out adaptive design methodology. We use the function `compute_single_level_loo_samples` to do this. This function requires two inputs:
- A GP to find the new design point for (which we have)
- A `SimulatorDomain` object, which defines the input space on which the simulator inputs are defined.

The training data points read in all have simulator inputs that lie in the unit square. To define the corresponding `SimulatorDomain`, we need to provide the lower and upper bounds for each of the coordinates. In this case, there are two input coordinates and each is bounded by 0 and 1.

In [6]:
bounds = [(0, 1), (0, 1)]  # one pair of bounds for each input dimension
domain = SimulatorDomain(bounds)

As an aside, the `domain` object has a method for computing the 'pseudopoints' arising from a given set of simulator inputs. 'Pseudopoints' are used in the leave-one-out adaptive sampling methodology. To calculate the pseudopoints around the training inputs, we can do the following:

In [7]:
# Get the training inputs
inputs = [datum.input for datum in training_data]

# Compute the pseudopoints
domain.calculate_pseudopoints(inputs)

(Input(0, 0.2937336532118139),
 Input(1, 0.5099688760976656),
 Input(0.0608263616598308, 0),
 Input(0.3825924172249514, 1),
 Input(0, 0),
 Input(0, 1),
 Input(1, 0),
 Input(1, 1))

Now we can generate a new design point:

In [8]:
x = compute_single_level_loo_samples(gp, domain)
x

(Input(0.59428475107308, 3.4352338196264043e-09),)

If instead we wanted to compute a batch of training inputs in one go, we can do this by specifying a different batch size:

In [9]:
new_design_pts = compute_single_level_loo_samples(gp, domain, batch_size=5)
new_design_pts

(Input(0.5942186785675462, 5.930666568954024e-09),
 Input(0.9999999713693596, 0.1605399020561663),
 Input(0.3612356454533319, 9.263520350799581e-08),
 Input(0.7646070596242592, 0.9999999988547795),
 Input(5.669068658953336e-09, 0.8635272768870305))

Note how the new design points all lie within the simulator domain we defined earlier, i.e. they all lie in the unit square.

By default, the leave-one-out errors GP calculated during the adaptive sampling method uses a fresh copy of the supplied GP. In particular, it will use the same kernel function as the original. We can instead specify that a different GP is used by supplying a new one with the settings we desire. For example, to ensure that the leave-one-out errors GP uses a squared exponential kernel instead of a Matern 5/2, we can do the following:

In [10]:
sqexp_gp = MogpEmulator(kernel="SquaredExponential")
x = compute_single_level_loo_samples(gp, domain, loo_errors_gp=sqexp_gp)
x

Too few unique inputs; defaulting to flat priors
Too few unique inputs; defaulting to flat priors


(Input(0.6184889822479795, 5.040602379935422e-10),)